# Floodscan Processing — Riverine LGAs

Selects Floodscan pixels within 10 km of the Benue river that intersect the riverine LGAs, then processes the full daily COG archive (1998–2025) and writes a pixel-level parquet to blob storage. Set `STATE` below to switch between Adamawa and Benue state. Run this notebook once per state before running `12_riverine_flooding_analysis.ipynb`.

In [ ]:
import rioxarray  # noqa: F401 — registers .rio accessor
import xarray as xr
import pandas as pd
import ocha_stratus as stratus
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path
from shapely.geometry import box
from tqdm.auto import tqdm
from dask.diagnostics import ProgressBar

from src.datasources import hydrosheds
from src.datasources.glofas import GF_STATIONS
from src.constants import STATE_CONFIG

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

## State configuration

Set `STATE` to `"Benue"` or `"Adamawa"` to switch between the two areas of interest.

In [ ]:
STATE = "Benue"  # or "Adamawa"
cfg = STATE_CONFIG[STATE]

## Load geolayers

In [ ]:
gdf_lga = stratus.codab.load_codab_from_blob("NGA", admin_level=2)
gdf_rivers = hydrosheds.load_selected_rivers()

gdf_state = gdf_lga[gdf_lga[cfg["adm1_col"]] == cfg["adm1_val"]].copy()
gdf_intersecting = gdf_state[gdf_state["ADM2_PCODE"].isin(cfg["lga_pcodes"])]
gdf_non_intersecting = gdf_state.drop(index=gdf_intersecting.index)

gdf_benue_river = gdf_rivers[
    (gdf_rivers.geometry.centroid.x > cfg["river_x_min"]) & (gdf_rivers.geometry.centroid.y < 10)
].copy()
minx, _, maxx, _ = gdf_state.total_bounds
gdf_river_clipped = gpd.clip(gdf_benue_river, box(minx, -90, maxx, 90))

station = GF_STATIONS[cfg["glofas_station"]]

## LGA selection map

Selected riverine LGAs highlighted in blue, with the Benue river and GloFAS station.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

gdf_non_intersecting.plot(ax=ax, color="#F0F0F0", edgecolor="white", linewidth=0.5)
gdf_intersecting.plot(ax=ax, color="#007CE0", edgecolor="white", linewidth=0.5, alpha=0.8)
gdf_river_clipped.plot(ax=ax, color="#1E5A8E", linewidth=1.2, zorder=3)
ax.plot(station["lon"], station["lat"], marker="o", color="#F2645A", markersize=7, zorder=4)
ax.annotate(
    cfg["station_label"],
    xy=(station["lon"], station["lat"]),
    xytext=(6, 6), textcoords="offset points",
    fontsize=8, fontweight="bold", color="#F2645A",
    bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.8),
)
for _, row in gdf_intersecting.iterrows():
    ax.annotate(
        row["ADM2_EN"],
        xy=(row.geometry.centroid.x, row.geometry.centroid.y),
        fontsize=6, ha="center", color="white", fontweight="bold",
    )

ax.set_axis_off()
ax.set_title(f"Selected {STATE} State LGAs", fontsize=13, pad=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / f"{STATE.lower()}_lgas_river.png", bbox_inches="tight", dpi=150)
plt.show()

print(f"Selected LGAs ({len(gdf_intersecting)}):")
print(sorted(gdf_intersecting["ADM2_EN"].tolist()))

## Demo: pixel selection

Demonstrates the two-step pixel selection on a single Floodscan COG: first clip to a 10 km buffer around the Benue river, then retain only pixels touching the selected LGAs (`all_touched=True`).

In [ ]:
river_buffer_10km = (
    gdf_benue_river.to_crs("EPSG:3857")
    .buffer(10_000)
    .to_crs(4326)
    .union_all()
)

DEMO_DATE = "2022-09-20"
blob_name = f"floodscan/daily/v5/processed/aer_area_300s_v{DEMO_DATE}_v05r01.tif"
da = stratus.open_blob_cog(blob_name, stage="prod", container_name="raster")
da = da.sel(band=1).drop_vars("band")

da_clip = da.rio.clip([river_buffer_10km])
da_clip = da_clip.rio.clip(gdf_intersecting.geometry, all_touched=True)
n_pixels = int((~da_clip.isnull()).sum())

fig, ax = plt.subplots(figsize=(10, 8))
gdf_non_intersecting.plot(ax=ax, color="#F0F0F0", edgecolor="white", linewidth=0.5)
gdf_intersecting.plot(ax=ax, color="#007CE0", edgecolor="white", linewidth=0.5, alpha=0.3)
gdf_river_clipped.plot(ax=ax, color="#1E5A8E", linewidth=1.2, zorder=3)
xlim, ylim = ax.get_xlim(), ax.get_ylim()
vmax = max(float(da_clip.quantile(0.95)), 0.01)
da_clip.plot(
    ax=ax, cmap="YlOrRd", vmin=0, vmax=vmax,
    add_colorbar=True, cbar_kwargs={"label": "SFED", "shrink": 0.5, "pad": 0.02},
    zorder=4,
)
ax.set_xlim(xlim); ax.set_ylim(ylim)
ax.set_axis_off()
ax.set_title(
    f"Floodscan pixels within 10km of Benue River\n(selected LGAs only — {DEMO_DATE})",
    fontsize=12, pad=12,
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / f"{STATE.lower()}_fs_pixels_demo.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Pixels selected: {n_pixels}")

## Full processing

Loads every available Floodscan COG (1998-01-12 → 2025-12-31), applies the same two-step clip, and writes the resulting pixel-level dataframe to blob storage.

In [ ]:
blob_name_fmt = "floodscan/daily/v5/processed/aer_area_300s_v{date_str}_v05r01.tif"
dates = pd.date_range("1998-01-12", "2025-12-31")

das = []
for date in tqdm(dates, desc="Loading COGs"):
    da_in = stratus.open_blob_cog(
        blob_name_fmt.format(date_str=date.date()), stage="prod", container_name="raster"
    )
    da_in = da_in.sel(band=1).drop_vars("band")
    da_in["date"] = date
    das.append(da_in)

fs = xr.concat(das, dim="date")
fs_clip = fs.rio.clip([river_buffer_10km])
fs_clip = fs_clip.rio.clip(gdf_intersecting.geometry, all_touched=True)

with ProgressBar():
    df_out = (
        fs_clip.to_dataframe(name="SFED")
        .reset_index()
        .dropna(subset=["SFED"])
        [["date", "y", "x", "SFED"]]
    )

df_out["x"] = df_out["x"].round(4)
df_out["y"] = df_out["y"].round(4)

stratus.upload_parquet_to_blob(df_out, cfg["floodscan_blob"])

print(f"Saved {len(df_out):,} rows → {cfg['floodscan_blob']}")
print(f"Unique pixels: {df_out[['x', 'y']].drop_duplicates().shape[0]}")
print(f"Date range: {df_out['date'].min().date()} – {df_out['date'].max().date()}")